# Module 2 — Instrumenting a multi-step agent

**ODSC AI West 2026 · Production-Grade Agent & LLM Observability with Opik**

A single decorated function is a trace. A real agent is a **tree**, and the tree is where
the answers live. In this module you will instrument a retrieval → rerank → tool → generate
pipeline, then use its trace to answer a question that end-to-end timing gets *wrong*.

It ends with a **Your turn** exercise: catching a hallucination by reading its trace.

## Setup

In [ ]:
%pip install -q opik openai

In [ ]:
import os, sys

# Work from the repo root. A local Jupyter starts in notebooks/, so step up; Colab starts
# empty, so clone the repo first.
if not os.path.exists("workshop"):
    if os.path.exists("../workshop"):
        os.chdir("..")
    else:
        !git clone -q https://github.com/comet-ml/opik-odsc-west-2026.git
        os.chdir("opik-odsc-west-2026")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
import getpass

import opik

PROJECT = "odsc-workshop"
# Self-hosted Opik? Export OPIK_URL_OVERRIDE before starting Jupyter (see SETUP.md).
# Otherwise this connects to Opik Cloud and asks for your API key.
SELF_HOSTED_URL = os.environ.get("OPIK_URL_OVERRIDE")
opik.configure(use_local=bool(SELF_HOSTED_URL), url_override=SELF_HOSTED_URL, project_name=PROJECT)

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (Enter to continue without one): ")

os.environ["OPIK_PROJECT_NAME"] = PROJECT

## 1. The agent

It answers questions about Opik over a small document corpus. Four steps, each one a span:

| Step | Span type | What it does |
|---|---|---|
| `retrieve` | general | keyword search over 13 documents |
| `rerank` | llm | reorders candidates — **off by default** |
| `lookup_glossary` | tool | a deterministic dictionary lookup |
| `generate_answer` | llm | writes the answer from the retrieved context |

In [ ]:
import inspect

from workshop import agent

print(inspect.getsource(agent.answer_question))

Nothing here is observability code. It is four ordinary functions with a decorator each.

In [ ]:
result = agent.answer_question("Why does a trace need replacing semantics at all?")

print(result["answer"])
print()
print("retrieved:", result["doc_ids"])

Open the trace Opik printed. You should see `rag_agent` at the root with `retrieve`,
`lookup_glossary` and `generate_answer` nested beneath it — and, inside `generate_answer`,
a span the OpenAI integration added for you.

## 2. Is the reranker worth it?

A reranker is a second model call that reorders retrieved documents. It should cost
something. Let's find out how much — the obvious way first.

In [ ]:
import statistics
import time

import datetime as dt

QUESTION = "Why does a trace need replacing semantics at all?"
timing_started = dt.datetime.now(dt.timezone.utc)


def time_it(use_reranker, n=5):
    runs = []
    for _ in range(n):
        t0 = time.perf_counter()
        agent.answer_question(QUESTION, use_reranker=use_reranker)
        runs.append(time.perf_counter() - t0)
    return statistics.median(runs)


off = time_it(False)
on = time_it(True)
print(f"reranker off : {off:.2f}s")
print(f"reranker on  : {on:.2f}s")
print(f"ratio        : {on / off:.1f}x")

### The trap

Run that cell two or three times. The ratio will not hold still — we have measured the
same comparison at **1.4x** in one session and **1.0x** ("the reranker is free!") in the next.

Neither number is real. Run-to-run variance in `generate_answer` is 1.5–2.5 s, which is
*larger than the thing we are trying to measure*. Aggregate timing cannot see a component
smaller than its own noise.

This is the entire argument for spans.

## 3. Ask the trace instead

Every step has its own span with its own duration. So don't time the request — read the tree.

In [ ]:
opik.flush_tracker()

from opik.rest_api.core.api_error import ApiError


def wait_for(predicate, timeout=90, interval=3):
    """Ingestion is asynchronous and batched, so a freshly logged trace is not instantly
    queryable — and until the first one lands, its project does not exist yet either.
    Poll instead of guessing at a sleep."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            found = predicate()
        except ApiError as exc:
            if exc.status_code != 404:  # 404: the project hasn't been created yet
                raise
            found = None
        if found:
            return found
        time.sleep(interval)
    raise TimeoutError(f"nothing arrived after {timeout}s")


opik_client = opik.Opik()


# A reranked trace has 7 spans: two of them are the model calls the OpenAI wrapper records.
# Without an OpenAI key (canned mode) there are no model calls, so 5.
EXPECTED_SPANS = 5 if agent._IS_CANNED else 7


def find_reranked():
    # One span query rather than one per trace: span searches are rate-limited on Opik
    # Cloud, and a per-trace loop would spend most of its time waiting.
    rerank = [
        s for s in opik_client.search_spans(
            project_name=PROJECT, filter_string='name = "rerank"', max_results=20
        )
        if s.start_time >= timing_started  # only this run's traces, not earlier ones
    ]
    if not rerank:
        return None
    newest = max(rerank, key=lambda s: s.start_time)
    spans = opik_client.search_spans(
        project_name=PROJECT, trace_id=newest.trace_id, max_results=50
    )
    # Wait until every span has landed and closed.
    if len(spans) < EXPECTED_SPANS or any(s.duration is None for s in spans):
        return None
    return opik_client.get_trace_content(newest.trace_id), spans


trace, spans = wait_for(find_reranked, interval=5)
by_parent = {}
for s in spans:
    by_parent.setdefault(s.parent_span_id, []).append(s)


def show(parent=None, depth=0):
    for s in sorted(by_parent.get(parent, []), key=lambda x: x.start_time):
        cost = f"  ${s.total_estimated_cost:.8f}" if s.total_estimated_cost else ""
        print(f"{'  ' * depth}{s.name:<24} {s.type:<8} {s.duration:>9.2f} ms{cost}")
        show(s.id, depth + 1)


# Sum the spans rather than reading trace.total_estimated_cost: the trace-level
# rollup is computed after the spans land, so reading it immediately can show 0.
span_cost = sum(s.total_estimated_cost or 0 for s in spans)
print(f"TRACE {trace.name}  {trace.duration or 0:.2f} ms  ${span_cost:.8f}\n")
show()

### Two things that stopwatch could not tell you

**The reranker was never free.** It is a real, repeatable share of the request — roughly
570–690 ms every single time. End-to-end timing hid it inside its own variance.

**Retrieval is not your problem.** It lands around 0.1 ms, which is about **0.005%** of the
request. The instinct to go optimise the vector store is, here, measurably wrong. All of
the latency and all of the cost are in the model calls.

Note also where cost appears: on the innermost LLM span, not on the wrapper you decorated.
Cost belongs to the call that actually spent the money.

## 4. Make traces findable

In production you are not reading one trace, you are searching thousands. Tags and metadata
are how you get back to the ones that matter.

In [ ]:
from opik import opik_context, track


@track
def answer_with_context(question: str, tenant: str):
    opik_context.update_current_trace(
        tags=["odsc", "module-2", f"tenant:{tenant}"],
        metadata={"corpus_version": "v1", "reranker": False},
    )
    return agent.answer_question(question)


answer_with_context("What licence is Opik released under?", tenant="acme")
opik.flush_tracker()

tagged = wait_for(
    lambda: opik_client.search_traces(
        project_name=PROJECT,
        filter_string='tags contains "tenant:acme"',
        max_results=10,
    )
)
print(f"found {len(tagged)} trace(s) for tenant:acme")

## Your turn — catch a hallucination yourself

`data/eval_dataset.jsonl` holds five *adversarial* questions: their answers are almost, but
not quite, in the corpus.

1. Pick one (change `PICK` below) and ask the agent.
2. Open its trace. Which documents were retrieved — and does that context actually support
   the answer?
3. Find the span that carries the cost.

Not every adversarial question fools the agent every time — try two or three. That
variability is exactly why Module 4 measures this systematically instead of by eye.

In [ ]:
import datetime as dt
import json

adversarial = [
    row for row in map(json.loads, open("data/eval_dataset.jsonl"))
    if row["category"] == "adversarial_refusal"
]
for i, row in enumerate(adversarial):
    print(i, row["question"])

PICK = 3
question = adversarial[PICK]["question"]
asked_at = dt.datetime.now(dt.timezone.utc)
mine = agent.answer_question(question)

print(f"\nQ: {question}\nA: {mine['answer']}\nretrieved: {mine['doc_ids']}")

### Reference: the same reading, done programmatically

In [ ]:
opik.flush_tracker()


def my_trace():
    for t in opik_client.search_traces(project_name=PROJECT, max_results=20):
        if (t.input or {}).get("question") == question and t.start_time >= asked_at:
            return t
    return None


found = wait_for(my_trace)
for s in sorted(opik_client.search_spans(project_name=PROJECT, trace_id=found.id, max_results=50),
                key=lambda x: x.start_time):
    cost = f"  ${s.total_estimated_cost:.8f}" if s.total_estimated_cost else ""
    print(f"{s.name:<24} {s.type:<8}{cost}")

print("\nThe context the answer had to work with:")
for text in mine["context"]:
    print(" -", text.splitlines()[0].lstrip("# "))

## What you just did

- Instrumented a four-step agent without writing observability code
- Watched aggregate timing give the wrong answer twice, in opposite directions
- Used the span tree to get the right one
- Located where latency and cost actually concentrate — and where they don't
- Tagged traces so you can find them again

➡️ **Module 3 — `03_scale_and_analytics.ipynb`**: what happens to these traces once
there are millions of them.